# UltraGliClass stage 2 FULL run (arm C), session-aware

**What:** pilot arm C recipe (many-label expansion 20-150 labels + permutation-consistency) on the FULL stage-2 mix (`configs/data_stage2_full.yaml`, ~1.74M rows),
27,000 steps x 64 rows, init = the stage-1 model, seed 42. Fast settings measured on T4 x2: per-device batch 4 x accum 8, gradient checkpointing OFF, fp16,
length grouping OFF, `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`, DataLoader workers 0 (host-RAM safety).
**Partly in-domain:** 3,000 `LocalLLaMA/typed-decisions` TRAIN rows (customer_service + agent_trace_observability). massive and financial_phrasebank are NOT zero-shot for this model.

**Sessions (<= 11 h each, see STAGE2_FULL_JOB_SPEC.md):** session 1 = fresh (attach code dataset v6 + S1 output). Session 2+ = attach ALSO the previous version's
output of THIS kernel; `PREV_OUTPUT = "auto"` finds it and training resumes from the newest checkpoint (saved every 500 steps and at the session time cap).
At each session end: quick eval of the newest checkpoint -> `results/stage2_full_C_mid_step<N>.json`. At completion: final quick eval -> `results/stage2_full_C_final_quick.json`.
Stops (and reports) on NaN, OOM (after one smaller-micro-batch retry), stall (15 min no output -> kill, then one 1-GPU resume), or the session cap.


In [ ]:
# ---- knobs ----
ARMS          = ["full_C"]   # ONE arm: configs/stage2_full_C.yaml (do not change)
INIT_MODEL    = "auto"     # S1 final_model dir, OR the S1 kernel-output root, OR "auto" = search /kaggle/input for runs/stage1_ultrabert150m/final_model
DATA_RECIPE   = "configs/data_stage2_full.yaml"
DATA_DIR      = "data/stage2_full"
SEMANTIC_DATA_APPROVED = True   # Srimon approved the LocalLLaMA/typed-decisions TRAIN split (2 workflows, labelled in-domain), 24 Oct 2026
OOM_RETRY_SMALLER_BATCH = True  # on OOM: ONE resume with per-device batch 2 x accum 16 (same 64 rows/step); a 2nd OOM stops the session
PREP_ARGS     = ""                              # smoke tests: "--streaming"
CALIB_FILE    = "calib/stage1_val.jsonl"       # = data/stage1/val.jsonl used by the S1 and ModernBERT evals (same global-T fit)
CODE_INPUT    = "auto"     # auto-detect the code dataset under /kaggle/input
PREV_OUTPUT   = "auto"     # "auto" = previous version's output of THIS full run (runs/stage2_full_C) if attached, else fresh; None = force fresh
USE_SINGLE_GPU = False     # True = never launch 2 processes (python -u train.py on GPU 0, grad-accum doubled)
AUTO_FALLBACK_SINGLE_GPU = True   # 2-GPU run stalls/crashes (not OOM) => resume on 1 GPU in the same session
DIST_CHECK    = True       # 20-s NCCL pre-flight before the first 2-GPU launch
NCCL_P2P_DISABLE = False   # True = start with NCCL P2P off (auto-retried if the pre-flight hangs)
STALL_MINUTES = 15         # watchdog: no output for this long => kill the run
SESSION_HOURS = 12.0
RESERVE_HOURS = 0.5
ARM_MAX_HOURS = 10.6       # train budget per SESSION (train.py saves a checkpoint and exits needs_resume; next session resumes)
TOTAL_MAX_HOURS = 10.8     # STOP: session clock cap incl. setup, data, train, eval; + ~0.2 h output upload => <= 11 h (GPU-hours = session hours)
EVAL_RESERVE_HOURS = 0.2   # kept free for the quick eval after an arm
QUICK_EVAL    = True
EVAL_MAX_PER_DATASET = 500
EVAL_PERMS    = 3
EVAL_CALIB_N  = 2000             # rows of CALIB_FILE used to fit the global temperature (S1: 2000)
EVAL_TYPED_MAX_CASES = None      # None = all 400 cases / 2,000 decisions (as in S1)
DROP_CKPT_AFTER_ARM = True       # delete checkpoint-* only once the run is COMPLETE and its final eval is written (final_model is kept)
MID_EVAL      = True             # quick eval of the newest checkpoint at the end of an unfinished session
EXTRA_SET     = []         # dotted overrides for every arm, e.g. ["train.max_steps=20"]
ARM_SET       = {}         # per-arm overrides, e.g. {"C": ["data.consistency.pair_prob=0.1"]}
import os, time, glob, json, sys, shutil, subprocess
T0 = time.time()
LOG_DIR = "/kaggle/working/logs"; os.makedirs(LOG_DIR, exist_ok=True)
NB_LOG = open(os.path.join(LOG_DIR, f"notebook_{time.strftime('%Y%m%d_%H%M%S')}.log"), "a", buffering=1)
def say(*a):
    line = f"[nb {time.strftime('%H:%M:%S')} +{(time.time() - T0) / 60:.1f}m] " + " ".join(str(x) for x in a)
    print(line, flush=True); NB_LOG.write(line + "\n")

def find_code_input():
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.py", recursive=True)]
    hits = [h for h in hits if os.path.exists(os.path.join(h, "configs", "stage2_full_C.yaml"))]   # needs the v6 code (full-run configs)
    hits = [h for h in hits if not os.path.isdir(os.path.join(h, "runs"))] or hits
    hits.sort(key=lambda p: (0 if "ultragliclass-code" in p else 1, len(p)))
    say("code candidates:", hits)
    return hits[0] if hits else None

def find_prev_output():
    hits = sorted({p.split("/runs/")[0] for p in glob.glob("/kaggle/input/**/runs/stage2_full_C/STATUS.json", recursive=True)}, key=len)
    say("previous full-run output candidates:", hits)
    return hits[0] if hits else None

def find_init_model():
    cands = []
    if INIT_MODEL != "auto":
        cands = [INIT_MODEL] if os.path.exists(os.path.join(INIT_MODEL, "config.json")) else \
            [os.path.dirname(p) for p in glob.glob(os.path.join(INIT_MODEL, "**", "final_model", "config.json"), recursive=True)]
    else:
        cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/runs/stage1_ultrabert150m/final_model/config.json", recursive=True)]
    cands.sort(key=len)
    say("S1 model candidates:", cands)
    return cands[0] if cands else None

if CODE_INPUT == "auto":
    CODE_INPUT = find_code_input()
if PREV_OUTPUT == "auto":
    PREV_OUTPUT = find_prev_output()
INIT_SRC = find_init_model()
assert ARMS == ["full_C"], ARMS
say("CODE_INPUT =", CODE_INPUT, "| INIT_SRC =", INIT_SRC, "| PREV_OUTPUT =", PREV_OUTPUT, "| ARMS =", ARMS, "| USE_SINGLE_GPU =", USE_SINGLE_GPU)
assert CODE_INPUT and os.path.exists(os.path.join(CODE_INPUT, "configs", "stage2_full_C.yaml")), "v6 code dataset (with configs/stage2_full_C.yaml) not found under /kaggle/input"
assert INIT_SRC, "S1 model not found: attach the output of kernel ultragliclass-s1 (runs/stage1_ultrabert150m/final_model) or set INIT_MODEL"


In [ ]:
import shutil, subprocess
WORK = "/kaggle/working/ultragliclass"
if not os.path.exists(os.path.join(WORK, "train.py")):
    shutil.copytree(CODE_INPUT, WORK, dirs_exist_ok=True, copy_function=shutil.copyfile,
                    ignore=shutil.ignore_patterns("runs", "data", "smoke_logs", "results", "__pycache__"))
subprocess.run(f"chmod -R u+w {WORK}", shell=True)  # copytree copies read-only dir modes from /kaggle/input
os.chdir(WORK)
say("work dir", WORK)
torch_pin = subprocess.run("pip freeze | grep -E '^torch=='", shell=True, capture_output=True, text=True).stdout.strip()
open("/tmp/constraints.txt", "w").write(torch_pin + "\n")
say("torch pin:", torch_pin)
subprocess.run("pip install -q -c /tmp/constraints.txt -r requirements.txt", shell=True, check=True)
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)
import torch, transformers
NGPU = torch.cuda.device_count()
say("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", NGPU, "transformers", transformers.__version__)

In [ ]:
# The S1 model: validate it, then COPY it out of /kaggle/input (plain file copies, in this single process, before torchrun) so
# no rank ever reads the read-only input mount. INIT_LOCAL is what every arm gets as model.init_from.
INIT_LOCAL = "/tmp/ugc_init_model"
need = ["config.json", "tokenizer.json", "tokenizer_config.json"]
for f in need:
    assert os.path.exists(os.path.join(INIT_SRC, f)), f"S1 model dir {INIT_SRC} lacks {f}"
weights = [f for f in os.listdir(INIT_SRC) if f.endswith((".safetensors", ".bin"))]
assert weights, f"no weight file in {INIT_SRC}: {os.listdir(INIT_SRC)}"
size = sum(os.path.getsize(os.path.join(INIT_SRC, f)) for f in os.listdir(INIT_SRC))
assert size > 200e6, f"S1 model dir is only {size / 1e6:.0f} MB: incomplete"
cfg_s1 = json.load(open(os.path.join(INIT_SRC, "config.json")))
say("S1 model:", INIT_SRC, f"{size / 2**20:.0f} MB", weights, "| architectures", cfg_s1.get("architectures"), "| max_num_classes", cfg_s1.get("max_num_classes"))
shutil.rmtree(INIT_LOCAL, ignore_errors=True); os.makedirs(INIT_LOCAL)
for f in os.listdir(INIT_SRC):
    s = os.path.join(INIT_SRC, f)
    if os.path.isfile(s):
        shutil.copyfile(s, os.path.join(INIT_LOCAL, f))
        assert os.path.getsize(os.path.join(INIT_LOCAL, f)) == os.path.getsize(s), f"copy size mismatch {f}"
subprocess.run(f"chmod -R u+w {INIT_LOCAL}", shell=True)
say("S1 model copied to", INIT_LOCAL, sorted(os.listdir(INIT_LOCAL)))

In [ ]:
# Restore the previous output. Runs HERE, in the single notebook process, BEFORE torchrun: no rank ever touches
# /kaggle/input. Plain file copies (no read-only mode bits from the input mount), then chmod u+w, then verify sizes.
def restore(src_root):
    for sub in ("runs", "data", "results"):
        src = os.path.join(src_root, sub)
        if not os.path.isdir(src):
            say("restore: no", sub, "in", src_root); continue
        n = b = 0; t = time.time()
        for dirpath, _, files in os.walk(src):
            rel = os.path.relpath(dirpath, src_root)
            os.makedirs(rel, exist_ok=True)
            for f in files:
                s, d = os.path.join(dirpath, f), os.path.join(rel, f)
                size = os.path.getsize(s)
                if not (os.path.exists(d) and os.path.getsize(d) == size):
                    shutil.copyfile(s, d)
                assert os.path.getsize(d) == size, f"size mismatch {d}"
                n += 1; b += size
                if size > 50e6: say(f"restore: copied {d} ({size / 2**20:.0f} MB)")
        say(f"restore: {sub}: {n} files, {b / 2**30:.2f} GB in {time.time() - t:.0f}s")
    subprocess.run("chmod -R u+w runs data results 2>/dev/null; true", shell=True)

if PREV_OUTPUT:
    say("restore from", PREV_OUTPUT)
    restore(PREV_OUTPUT)
for st in glob.glob("runs/*/STATUS.json"):
    s = json.load(open(st)); say("found", st, s.get("state"), "step", s.get("global_step"), "last", s.get("last_checkpoint"))
for c in sorted(glob.glob("runs/*/checkpoint-*")):
    say("checkpoint", c, sorted(os.listdir(c)))

In [ ]:
# Data: built once in session 1 (deterministic: recipe + seed 43), restored from the previous output afterwards.
import hashlib
sys.path.insert(0, WORK)
from ugc.config import load_config as _lc
assert os.path.exists("calib/stage2_pilot_val.jsonl"), "calib/stage2_pilot_val.jsonl missing from the code dataset"
assert SEMANTIC_DATA_APPROVED, "the full recipe contains LocalLLaMA/typed-decisions TRAIN rows (owner approval D10)"
os.makedirs("results", exist_ok=True)
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
    return h.hexdigest()
MD5_REC = "results/stage2_full_data_md5.json"
rebuilt = False
if not os.path.exists(os.path.join(DATA_DIR, "train.jsonl")):
    say("preparing data:", DATA_RECIPE, PREP_ARGS); t = time.time()
    subprocess.run(f"python -u prepare_data.py --recipe {DATA_RECIPE} {PREP_ARGS}", shell=True, check=True)
    rebuilt = True; say(f"data prep took {(time.time() - t) / 60:.1f} min")
else:
    say("data already present (restored from the previous output):", DATA_DIR)
DATA_MD5 = md5(os.path.join(DATA_DIR, "train.jsonl"))
if os.path.exists(MD5_REC):
    rec_md5 = json.load(open(MD5_REC))["train_md5"]
    assert rec_md5 == DATA_MD5, (f"STOP: train.jsonl md5 {DATA_MD5} != session-1 md5 {rec_md5}; the resume data order would differ. "
                                 "Attach the previous output that contains data/stage2_full (do not rebuild).")
    say("train.jsonl md5 matches session 1:", DATA_MD5, "(rebuilt)" if rebuilt else "")
else:
    json.dump({"train_md5": DATA_MD5, "recipe": DATA_RECIPE, "time": time.strftime("%Y-%m-%d %H:%M:%S")}, open(MD5_REC, "w"))
man = json.load(open(os.path.join(DATA_DIR, "manifest.json")))
shutil.copyfile(os.path.join(DATA_DIR, "manifest.json"), "results/stage2_full_data_manifest.json")
_t = _lc("configs/stage2_full_C.yaml", [f"model.init_from={INIT_LOCAL}"] + EXTRA_SET)["train"]
need_rows = int(_t["max_steps"]) * int(_t["per_device_train_batch_size"]) * int(_t.get("gradient_accumulation_steps", 1)) * \
    (NGPU if NGPU >= 2 and not USE_SINGLE_GPU else 1)      # = 27,000 x 64 = 1,728,000 on the real run
used = {k: v for k, v in man["sources"].items() if v.get("train_rows", 0) > 0}
bad = [k for k in used if any(x in k.lower() for x in ("dbpedia", "bitext", "banking77", "clinc", "massive", "snips", "phrasebank", "sst5"))]
sem = {k: v["train_rows"] for k, v in used.items() if "localllama/typed-decisions" in k.lower()}
proc_dedicated = sum(v["train_rows"] for k, v in used.items() if k.startswith("tasksource/procedural-typed-decisions"))
n_proc = n_tot = 0
with open(os.path.join(DATA_DIR, "train.jsonl")) as f:
    for line in f:
        n_tot += 1; n_proc += ("procedural-typed-decisions" in json.loads(line)["source"])
PROC_SHARE = round(n_proc / max(n_tot, 1), 4)
val_txt = {json.loads(l)["text"] for l in open("calib/stage2_pilot_val.jsonl")}
n_val_in_train = 0
with open(os.path.join(DATA_DIR, "train.jsonl")) as f:
    for line in f:
        n_val_in_train += json.loads(line)["text"] in val_txt
say("train rows", man["train_rows"], "need", need_rows, "| val (own) rows", man["val_rows"], "| pools", man.get("taxonomy", {}).get("pool_sizes"),
    "| labels/row", man["labels_per_row"])
say(f"procedural-style rows {n_proc} ({100 * PROC_SHARE:.2f} %), dedicated procedural rows {proc_dedicated} | typed-decisions TRAIN rows {sem} | "
    f"train rows whose text equals a val text: {n_val_in_train}")
say("sources used:", {k: v["train_rows"] for k, v in used.items()})
print(json.dumps({k: v.get("license") for k, v in used.items()}, indent=1))
json.dump({"train_rows": man["train_rows"], "need_rows": need_rows, "procedural_share": PROC_SHARE, "procedural_rows": n_proc,
           "semantic_rows": sem, "train_rows_text_in_val": n_val_in_train, "md5": DATA_MD5,
           "licenses": {k: v.get("license") for k, v in used.items()}}, open("results/stage2_full_data_check.json", "w"), indent=1)
assert not bad, f"excluded / eval-set source in the data: {bad}"
assert man["train_rows"] >= need_rows, f"only {man['train_rows']} train rows, need {need_rows}"
assert proc_dedicated == 0, f"dedicated procedural rows present: {proc_dedicated}"
assert sorted(k.split(":")[1] for k in sem) == ["agent_trace_observability", "customer_service"] and sum(sem.values()) == 3000, sem
assert 0.035 <= PROC_SHARE <= 0.065 or man["train_rows"] < 100_000, f"procedural share {PROC_SHARE} not at the E1 level (~0.049)"
assert n_val_in_train <= 0.01 * len(val_txt), f"{n_val_in_train} train rows share a text with the val set"


In [ ]:
# Launch with live, unbuffered streaming to this notebook AND to /kaggle/working/logs/, a no-output watchdog, a NCCL pre-flight,
# a single-GPU fallback, then the quick eval, one arm after the other.
import signal, threading, queue, re
sys.path.insert(0, WORK)
from ugc.config import load_config
LAST_OOM = False

def run_streamed(cmd, env_extra, log_name, stall_min):
    global LAST_OOM
    LAST_OOM = False
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
           "TORCH_NCCL_DUMP_ON_TIMEOUT": "1", "NCCL_DEBUG": "WARN", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", **env_extra}
    path = os.path.join(LOG_DIR, f"{log_name}_{time.strftime('%H%M%S')}.log")
    say("RUN", cmd, "| env", env_extra, "| log", path)
    p = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, start_new_session=True)
    q = queue.Queue()
    threading.Thread(target=lambda: ([q.put(l) for l in p.stdout], q.put(None)), daemon=True).start()
    last, stalled = time.time(), False
    with open(path, "a", buffering=1) as f:
        while True:
            try:
                line = q.get(timeout=10)
            except queue.Empty:
                if time.time() - last > stall_min * 60:
                    say(f"WATCHDOG: no output for {stall_min} min -> killing process group"); stalled = True
                    os.killpg(p.pid, signal.SIGTERM); time.sleep(30)
                    if p.poll() is None: os.killpg(p.pid, signal.SIGKILL)
                    break
                continue
            if line is None: break
            if re.search(r"OutOfMemoryError|CUDA out of memory", line): LAST_OOM = True
            last = time.time(); print(line, end="", flush=True); f.write(line)
    rc = p.wait()
    say("EXIT", rc, "stalled" if stalled else "", "OOM" if LAST_OOM else "")
    return rc, stalled

def elapsed_h():
    return (time.time() - T0) / 3600

def arm_budget_h():
    return min(ARM_MAX_HOURS, TOTAL_MAX_HOURS - elapsed_h() - EVAL_RESERVE_HOURS, SESSION_HOURS - elapsed_h() - RESERVE_HOURS)

def arm_sets(arm):
    return [f"model.init_from={INIT_LOCAL}"] + EXTRA_SET + ARM_SET.get(arm, [])

SMALL_BATCH = False
def train_cmd(arm, single):
    sets = [f"train.time_budget_hours={max(0.1, arm_budget_h()):.2f}"] + arm_sets(arm)
    if SMALL_BATCH:   # OOM retry: half the micro-batch, double the accumulation (same rows per optimizer step)
        _c = load_config("configs/stage2_full_C.yaml", arm_sets(arm))["train"]
        sets += [f"train.per_device_train_batch_size={int(_c['per_device_train_batch_size']) // 2}", f"train.gradient_accumulation_steps={int(_c['gradient_accumulation_steps']) * 2}"]
    accum = int(load_config(f"configs/stage2_full_C.yaml", arm_sets(arm))["train"].get("gradient_accumulation_steps", 1)) * (2 if SMALL_BATCH else 1)
    if single and NGPU > 1:
        sets = [s for s in sets if not s.startswith("train.gradient_accumulation_steps=")] + [f"train.gradient_accumulation_steps={accum * NGPU}"]
    launcher = "python -u" if single else f"torchrun --standalone --nproc_per_node {NGPU}"
    return f"{launcher} train.py --config configs/stage2_full_C.yaml --resume auto --set {' '.join(sets)}"

single = USE_SINGLE_GPU or NGPU < 2
nccl_env = {"NCCL_P2P_DISABLE": "1"} if NCCL_P2P_DISABLE else {}
if not single and DIST_CHECK:
    rc, st = run_streamed(f"torchrun --standalone --nproc_per_node {NGPU} scripts/dist_check.py", nccl_env, "dist_check", 3)
    if rc != 0 and not nccl_env:
        say("pre-flight failed: retrying with NCCL_P2P_DISABLE=1"); nccl_env = {"NCCL_P2P_DISABLE": "1"}
        rc, st = run_streamed(f"torchrun --standalone --nproc_per_node {NGPU} scripts/dist_check.py", nccl_env, "dist_check_p2poff", 3)
    if rc != 0:
        say("pre-flight failed twice: using 1 GPU"); single = True

def read_status(out_dir):
    p = os.path.join(out_dir, "STATUS.json")
    return json.load(open(p)) if os.path.exists(p) else {}

def quick_eval(model_dir, res_path, tag):
    cmd = (f"python -u eval.py --model {model_dir} --suites gliclass intent typed --max_per_dataset {EVAL_MAX_PER_DATASET} "
           f"--permutations {EVAL_PERMS} --calib_file {CALIB_FILE} --calib_n {EVAL_CALIB_N} --max_length 1024 --out {res_path}" +
           (f" --typed_max_cases {EVAL_TYPED_MAX_CASES}" if EVAL_TYPED_MAX_CASES else ""))
    rc, _ = run_streamed(cmd, {}, f"quick_eval_{tag}", 30)
    if not os.path.exists(res_path):
        return {"eval_rc": rc}
    r = json.load(open(res_path)); d = r["datasets"]; t1 = (r.get("typed_decisions") or {}).get("T1", {})
    return {"eval_rc": rc, "avg_macro_f1": r.get("gliclass_avg_macro_f1"), "global_T": r.get("global_T"), "typed_acc": t1.get("accuracy"),
            "typed_by_type": t1.get("accuracy_by_type"), "typed_by_workflow": t1.get("accuracy_by_workflow"),
            "flip_many_label": {k: d[k]["perm_flip_rate"] for k in ("banking77", "clinc150", "massive", "SetFit/20_newsgroups") if k in d},
            "NOT_zero_shot": ["massive (HWU64 shares 56/59 intents)", "financial_phrasebank (fin-news sentiment, same 3 labels)",
                              "typed-decisions customer_service + agent_trace_observability (trained on their TRAIN split)"]}

def run_arm(arm):
    global single, SMALL_BATCH
    cfg = load_config("configs/stage2_full_C.yaml", arm_sets(arm))
    out_dir = cfg["train"]["output_dir"]
    st0 = read_status(out_dir)
    rec = {"arm": arm, "out_dir": out_dir, "state": None, "t0_hours": round(elapsed_h(), 3), "start_step": st0.get("global_step", 0)}
    st = st0
    if st.get("state") == "complete" and os.path.isdir(st.get("final_model", "")):
        say("FULL RUN already complete (restored): skipping training")
    else:
        if arm_budget_h() < 0.4:
            rec["state"] = "not_started_budget"; say(f"NOT STARTED, only {arm_budget_h():.2f} h of budget left"); return rec
        say(f"===== FULL arm C: step {rec['start_step']} -> {cfg['train']['max_steps']}, session train budget {arm_budget_h():.2f} h =====")
        rc, stalled = run_streamed(train_cmd(arm, single), ({"CUDA_VISIBLE_DEVICES": "0"} if single else nccl_env),
                                   f"train_{arm}_{'1gpu' if single else 'ddp'}", STALL_MINUTES)
        st = read_status(out_dir)
        if LAST_OOM and OOM_RETRY_SMALLER_BATCH and not SMALL_BATCH and st.get("state") != "complete" and arm_budget_h() >= 0.4:
            say("OOM -> ONE resume from the newest checkpoint with per-device batch 2 x accum 16 (same 64 rows/step)")
            SMALL_BATCH = True; rec["oom_retry_small_batch"] = True
            rc, stalled = run_streamed(train_cmd(arm, single), ({"CUDA_VISIBLE_DEVICES": "0"} if single else nccl_env),
                                       f"train_{arm}_smallbatch", STALL_MINUTES)
            st = read_status(out_dir)
        if (rc != 0 or stalled) and not LAST_OOM and not single and AUTO_FALLBACK_SINGLE_GPU and st.get("state") not in ("complete", "nan_stop") \
                and arm_budget_h() >= 0.4:
            say("2-GPU run failed/stalled -> resuming on 1 GPU from the newest checkpoint")
            single = True
            rc, stalled = run_streamed(train_cmd(arm, True), {"CUDA_VISIBLE_DEVICES": "0"}, f"train_{arm}_1gpu_fallback", STALL_MINUTES)
            st = read_status(out_dir)
        rec["oom"] = bool(LAST_OOM)
    hb = os.path.join(out_dir, "HEARTBEAT.json")
    if os.path.exists(hb): say("last heartbeat:", open(hb).read()[:300])
    rec["state"] = st.get("state", "no_status"); rec["status"] = st; rec["end_step"] = st.get("global_step")
    print(json.dumps(st, indent=2))
    if rec["state"] == "complete":
        res_path = "results/stage2_full_C_final_quick.json"
        if QUICK_EVAL and not os.path.exists(res_path):
            rec["eval"] = quick_eval(st["final_model"], res_path, "final")
        elif os.path.exists(res_path):
            rec["eval"] = {"note": "final eval already present", "path": res_path}
        if DROP_CKPT_AFTER_ARM and os.path.exists(res_path):
            for c in glob.glob(os.path.join(out_dir, "checkpoint-*")):
                shutil.rmtree(c, ignore_errors=True)
    else:
        say(f"FULL RUN: state={rec['state']} at step {rec['end_step']} (needs_resume = session cap reached -> attach this output and run the next session; "
            f"nan_stop = NaN loss; running/no_status = crash/OOM/stall -> report)")
        ck = sorted(glob.glob(os.path.join(out_dir, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[1]))
        if MID_EVAL and QUICK_EVAL and ck and rec["state"] == "needs_resume" and SESSION_HOURS - elapsed_h() > 0.3:
            step = int(ck[-1].rsplit("-", 1)[1])
            rec["eval"] = quick_eval(ck[-1], f"results/stage2_full_C_mid_step{step}.json", f"mid_step{step}")
            rec["eval"]["checkpoint"] = ck[-1]
    rec["wall_hours"] = round(elapsed_h() - rec["t0_hours"], 3)
    return rec

SESSION_IDX = 1 + len(glob.glob("results/stage2_full_summary_s*.json"))
SUMMARY = []
def write_summary():
    json.dump({"session": SESSION_IDX, "session_hours": round(elapsed_h(), 3), "arms": SUMMARY, "init_model_src": INIT_SRC, "prev_output": PREV_OUTPUT,
               "data_md5": DATA_MD5, "procedural_share": PROC_SHARE, "partly_in_domain": True},
              open(f"results/stage2_full_summary_s{SESSION_IDX}.json", "w"), indent=2)

say(f"SESSION {SESSION_IDX} of the full run")
rec = run_arm("full_C"); SUMMARY.append(rec); write_summary()
say(f"SESSION {SESSION_IDX} done: state={rec['state']} steps {rec.get('start_step')} -> {rec.get('end_step')} wall={rec.get('wall_hours')} h eval={rec.get('eval')}")

In [ ]:
print(json.dumps({"session": SESSION_IDX, "session_hours": round(elapsed_h(), 2), "arms": [{k: v for k, v in r.items() if k != "status"} for r in SUMMARY]}, indent=2))
for r in SUMMARY:
    s = r.get("status", {})
    say(f"FULL C: {r['state']} step {s.get('global_step')}/{s.get('max_steps')} train_loss {s.get('train_loss')} "
        f"final_eval_loss {(s.get('final_eval') or {}).get('eval_loss')} peak_mem {(s.get('final_eval') or {}).get('peak_mem_gb')} avg_F1 {(r.get('eval') or {}).get('avg_macro_f1')}")
if any(r["state"] != "complete" for r in SUMMARY):
    say("NOT COMPLETE: save this version; next session: attach code v6 + S1 output + THIS version's output, keep PREV_OUTPUT='auto', run all.")

In [ ]:
for out in sorted(glob.glob("runs/stage2_full_*")):          # unfinished arms keep their newest checkpoint only
    ck = sorted(glob.glob(os.path.join(out, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[1]))
    for c in ck[:-1]:
        shutil.rmtree(c)
subprocess.run("du -sh runs data results /kaggle/working/logs 2>/dev/null", shell=True)
say("session hours used:", round(elapsed_h(), 2)); NB_LOG.close()